# AI Investment Risk Radar: End-to-End Execution Pipeline

This notebook executes the end-to-end AI Investment Risk Radar pipeline by calling modular library functions from `ai_risk_radar`. It reproduces data loading, causal feature generation, model training, Platt scaling calibration, delayed trading simulation, and latest risk inference.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Add project root to sys.path
root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from ai_risk_radar.config import load_settings
from ai_risk_radar.data import load_snapshot
from ai_risk_radar.features import build_features
from ai_risk_radar.models import fit_models
from ai_risk_radar.simulation import simulate, weekly_targets
from ai_risk_radar.splits import make_splits

settings = load_settings(root_dir / "config.json")
print(f"Config loaded: horizon={settings.target_horizon}d, threshold={settings.target_threshold}")


## 1. Load Dataset Snapshot & Verify Integrity

In [ ]:
snapshot_dir = root_dir / settings.raw_dir / "snapshot"
bundle = load_snapshot(snapshot_dir)
print(f"Loaded {len(bundle.sessions)} trading sessions from {bundle.sessions[0].date()} to {bundle.sessions[-1].date()}")
print(f"Snapshot hashes: {bundle.manifest['hashes']}")


## 2. Feature Generation & Purged Splits

In [ ]:
features = build_features(bundle, settings)
splits = make_splits(features, settings)
print(f"Train sessions: {len(splits.train)}")
print(f"Calibration sessions: {len(splits.calibration)}")
print(f"Validation sessions: {len(splits.validation)}")
print(f"Test sessions (untouched): {len(splits.test)}")


## 3. Train & Calibrate Random Forest Downside Predictor

In [ ]:
models = fit_models(features, splits, settings)
rf_bundle = models["random_forest"]
print(f"Trained Random Forest: {rf_bundle.metadata['model_name']}")
print(f"Selected params: max_depth={rf_bundle.metadata.get('selected_depth')}, min_samples_leaf={rf_bundle.metadata.get('selected_min_leaf')}")
print(f"Probability status: {rf_bundle.metadata.get('probability_status')}")


## 4. Run Delayed Simulation on Validation Set

In [ ]:
df = features.frame
val_prices = bundle.adjusted_close.loc[splits.validation]
val_X = df.loc[splits.validation, list(features.feature_columns)]

# Generate calibrated predictions on validation
val_raw_p = rf_bundle.estimator.predict_proba(val_X)[:, 1]
if rf_bundle.calibrator is not None:
    val_cal_p = rf_bundle.calibrator.predict_proba(val_raw_p)
else:
    val_cal_p = val_raw_p

val_probs = pd.Series(val_cal_p, index=splits.validation)

# Generate weekly targets for threshold 0.50
targets = weekly_targets(
    val_probs,
    val_prices,
    settings,
    threshold=0.50,
)

# Run delayed execution simulation
sim_res = simulate(val_prices, targets, settings, cost_rate=settings.cost_rate)
metrics = sim_res.metrics

print("Validation Simulation Performance (Threshold 0.50):")
print(f"  CAGR: {metrics['cagr']:.1%}")
print(f"  Max Drawdown: {metrics['max_drawdown']:.1%}")
print(f"  Sharpe Ratio (rf=0): {metrics['sharpe']:.2f}")
print(f"  Turnover: {metrics['turnover']:.2f}x")
print(f"  Defensive Occupancy: {metrics['defensive_occupancy']:.1%}")


## 5. Latest Market Risk Inference

Inference on the latest completed market session without fitting models or evaluating future outcomes.


In [ ]:
latest_session = df.index[-1]
latest_X = df.loc[[latest_session], list(features.feature_columns)]
latest_raw_p = float(rf_bundle.estimator.predict_proba(latest_X)[0, 1])

if rf_bundle.calibrator is not None:
    latest_cal_p = float(rf_bundle.calibrator.predict_proba(np.array([latest_raw_p]))[0])
else:
    latest_cal_p = latest_raw_p

risk_level = "High" if latest_cal_p >= 0.50 else ("Medium" if latest_cal_p >= 0.35 else "Low")

print(f"Latest Market Session: {latest_session.strftime('%Y-%m-%d')}")
print(f"Downside Risk Probability: {latest_cal_p:.1%}")
print(f"Risk Level: {risk_level}")
if latest_cal_p >= 0.50:
    print("Suggested Stance: Defensive (SOXX 10%, XLK 15%, SPY 35%, SHY 40%)")
else:
    print("Suggested Stance: Growth (SOXX 35%, XLK 35%, SPY 20%, SHY 10%)")
